# DeBERTa + Cryptic Wordplay — Simple Inference Playground

This notebook is intentionally simple.

There is **no fine-tuning** and **no training loop**.

We load a pretrained DeBERTa model from Hugging Face and use it to explore:

- Wordplay examples from the `mdda/cryptic-wordplay` dataset
- tokenization
- hidden-state representations
- sentence embeddings using mean pooling
- cosine similarity between a clue/answer and different candidate reasonings

Important: the similarity values in this notebook are **not trained reasoning-correctness scores**.
They are only a way to inspect how the pretrained model represents the texts.

## 1. Install dependencies

In Google Colab, you can optionally choose **Runtime → Change runtime type → GPU**.
For this small inference notebook, CPU also works.

In [1]:
!pip install -q -U transformers datasets sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.7 MB/s eta 0:00:00


## 2. Imports and model configuration

In [ ]:
import torch
import torch.nn.functional as F

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "microsoft/deberta-v3-small"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU visible - check the --gres=gpu request in the SLURM script")

## 3. Load the pretrained DeBERTa model from Hugging Face

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)
model.eval()

print("Loaded:", MODEL_NAME)
print("Hidden size:", model.config.hidden_size)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  286MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: microsoft/deberta-v3-small
Hidden size: 768


## 4. Load Wordplay examples from GitHub

The dataset contains fields such as:

- `clue`
- `answer`
- `pattern`
- `wordplay`

The clue marks the definition span with `{...}`.
We remove the braces when using the clue as model input.

In [ ]:
import os

BASE_URL = "https://raw.githubusercontent.com/mdda/cryptic-wordplay/main/prebuilt"

# Compute nodes are often behind a firewall, so prefer the local copy of the
# prebuilt files when it exists and fall back to GitHub otherwise.
LOCAL_PREBUILT = os.environ.get("PREBUILT_DIR", "cryptic-wordplay-main/prebuilt")

if os.path.isdir(LOCAL_PREBUILT):
    BASE = LOCAL_PREBUILT
else:
    BASE = BASE_URL

print("Loading data from:", BASE)

data_files = {
    "train": f"{BASE}/sample_teacow_train.jsonl",
    "validation": f"{BASE}/sample_teacow_val.jsonl",
}

dataset = load_dataset("json", data_files=data_files)

print(dataset)
print()
print(dataset["train"][0])

## 5. Look at a few Wordplay examples

In [5]:
def clean_clue(clue):
    return str(clue).replace("{", "").replace("}", "")


for i in range(5):
    ex = dataset["train"][i]

    print("CLUE:     ", clean_clue(ex["clue"]))
    print("ANSWER:   ", ex["answer"])
    print("WORDPLAY: ", ex["wordplay"])
    print("PATTERN:  ", ex["pattern"])
    print("-" * 100)

CLUE:      Composed , lawyer heading for tribunal in diocese
ANSWER:    SEDATE
WORDPLAY:  DA (lawyer) + T[ribunal] in SEE (diocese)
PATTERN:   6
----------------------------------------------------------------------------------------------------
CLUE:      A worried couple endlessly ringing about a place in Mexico
ANSWER:    ACAPULCO
WORDPLAY:  A + *(COUPL[E]) (*worried) ringing (around) CA (about, circa )
PATTERN:   8
----------------------------------------------------------------------------------------------------
CLUE:      Showing intelligence by keeping control in the auditorium
ANSWER:    BRAINY
WORDPLAY:  BY around (keeping) RAIN (control = “rein” – sounds like, in the auditorium)
PATTERN:   6
----------------------------------------------------------------------------------------------------
CLUE:      Hush-hush operations in Crete upset head of tourism
ANSWER:    TOP SECRET
WORDPLAY:  OPS (operations) in *(CRETE) (*upset) + T[ourism]
PATTERN:   3-6
--------------------------

## 6. Tokenization

Let's see how DeBERTa splits one clue into tokens.

In [6]:
example = dataset["train"][0]

clue = clean_clue(example["clue"])

encoded = tokenizer(
    clue,
    return_tensors="pt",
)

token_ids = encoded["input_ids"][0]

tokens = tokenizer.convert_ids_to_tokens(token_ids)

print("Text:")
print(clue)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids.tolist())

Text:
Composed , lawyer heading for tribunal in diocese

Tokens:
['[CLS]', '▁Composed', '▁,', '▁lawyer', '▁heading', '▁for', '▁tribunal', '▁in', '▁diocese', '[SEP]']

Token IDs:
[1, 64192, 366, 3710, 4051, 270, 24072, 267, 30019, 2]


## 7. Run one text through DeBERTa

`last_hidden_state` contains one vector for every token.

If the input has `T` tokens and the model hidden size is `H`, the shape is:

\[
[1, T, H]
\]

In [7]:
inputs = tokenizer(
    clue,
    return_tensors="pt",
    truncation=True,
    max_length=256,
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

hidden_states = outputs.last_hidden_state

print("Hidden-state shape:", hidden_states.shape)

Hidden-state shape: torch.Size([1, 10, 768])


## 8. Inspect token representations

Each token gets its own hidden vector.
Below we print the token together with the first few values of its vector.

In [8]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, vector in zip(tokens, hidden_states[0]):
    print(f"{token:20s} -> {vector[:5].cpu().numpy()}")

[CLS]                -> [-0.0041  -0.01564  0.0124  -0.00793  0.0195 ]
▁Composed            -> [-0.317  -0.3716  0.678   0.5063  0.2194]
▁,                   -> [-0.1066   0.3323   0.727    0.1549   0.05338]
▁lawyer              -> [ 1.134  -0.4739  0.312   0.2328 -0.1471]
▁heading             -> [ 0.1855  -0.01005  0.2267  -0.1246   0.3372 ]
▁for                 -> [-0.395   -0.1787  -0.08014 -0.4023  -0.2477 ]
▁tribunal            -> [ 0.013626  0.0155    0.6772   -0.1517   -0.006123]
▁in                  -> [-0.936    -0.002935  0.3262    0.3005   -0.04276 ]
▁diocese             -> [-0.281  -0.129   0.8877 -0.5327  0.0775]
[SEP]                -> [-0.014626 -0.01419   0.02032  -0.00504   0.01933 ]


## 9. Convert an entire text to one embedding

DeBERTa itself returns token-level representations.

For a simple sentence-level representation, we use **mean pooling**:

\[
h_{text}
=
\frac{1}{T}
\sum_{t=1}^{T} h_t
\]

Padding tokens are ignored.

In [9]:
def get_embedding(text, max_length=256):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    hidden = outputs.last_hidden_state
    mask = inputs["attention_mask"].unsqueeze(-1)

    summed = (hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1)

    embedding = summed / counts

    return embedding

In [10]:
embedding = get_embedding(clue)

print("Embedding shape:", embedding.shape)
print("First 10 values:")
print(embedding[0, :10])

Embedding shape: torch.Size([1, 768])
First 10 values:
tensor([-0.0721, -0.0848,  0.3787, -0.0030,  0.0282,  0.5469, -0.0649, -0.4250,
        -0.4429,  0.3813], device='cuda:0', dtype=torch.float16)


## 10. Cosine similarity

We can compare two embeddings using cosine similarity:

\[
\cos(h_1,h_2)
=
\frac{h_1 \cdot h_2}
{\|h_1\|\|h_2\|}
\]

Again, this is only a representation-similarity heuristic.
It is **not** a trained measure of reasoning correctness.

In [11]:
def cosine_similarity(text1, text2):
    emb1 = get_embedding(text1)
    emb2 = get_embedding(text2)

    score = F.cosine_similarity(emb1, emb2)

    return score.item()

## 11. Compare a clue+answer with a correct and incorrect reasoning

In [12]:
example = dataset["validation"][0]
wrong_example = dataset["validation"][1]

clue = clean_clue(example["clue"])
answer = example["answer"]

correct_reasoning = example["wordplay"]
wrong_reasoning = wrong_example["wordplay"]

context = f"CLUE: {clue}\nANSWER: {answer}"

correct_similarity = cosine_similarity(
    context,
    correct_reasoning
)

wrong_similarity = cosine_similarity(
    context,
    wrong_reasoning
)

print("CLUE:")
print(clue)

print("\nANSWER:")
print(answer)

print("\nCORRECT REASONING:")
print(correct_reasoning)
print("Similarity:", correct_similarity)

print("\nWRONG REASONING:")
print(wrong_reasoning)
print("Similarity:", wrong_similarity)

CLUE:
Fortune tellers scathing about Oscar

ANSWER:
TAROT

CORRECT REASONING:
TART (scathing) about O[scar] (phonetic alphabet)
Similarity: 0.90625

WRONG REASONING:
*(GIVE GEAR) (*out)
Similarity: 0.9033203125


## 12. A small convenience function

This function compares one candidate reasoning with the clue and answer.

In [13]:
def reasoning_similarity(clue, answer, reasoning):
    clue = clean_clue(clue)

    context = (
        f"CLUE: {clue}\n"
        f"ANSWER: {answer}"
    )

    return cosine_similarity(
        context,
        reasoning
    )

In [14]:
ex = dataset["validation"][2]

score = reasoning_similarity(
    clue=ex["clue"],
    answer=ex["answer"],
    reasoning=ex["wordplay"],
)

print("Clue:", clean_clue(ex["clue"]))
print("Answer:", ex["answer"])
print("Reasoning:", ex["wordplay"])
print("Similarity:", score)

Clue: Tumbler has cook taking drug
Answer: BEAKER
Reasoning: BAKER (cook) taking E (drug)
Similarity: 0.830078125


## 13. Compare several candidate reasonings

This lets you "play" with the pretrained model by ranking several explanations
according to embedding similarity.

In [15]:
def rank_reasonings(clue, answer, candidate_reasonings):
    scored = []

    for reasoning in candidate_reasonings:
        score = reasoning_similarity(
            clue,
            answer,
            reasoning
        )

        scored.append((reasoning, score))

    return sorted(
        scored,
        key=lambda x: x[1],
        reverse=True
    )

In [16]:
ex = dataset["validation"][0]

candidates = [
    ex["wordplay"],
    dataset["validation"][1]["wordplay"],
    dataset["validation"][2]["wordplay"],
]

ranking = rank_reasonings(
    clue=ex["clue"],
    answer=ex["answer"],
    candidate_reasonings=candidates,
)

print("CLUE:", clean_clue(ex["clue"]))
print("ANSWER:", ex["answer"])
print()

for i, (reasoning, score) in enumerate(ranking, start=1):
    print(f"{i}. similarity = {score:.4f}")
    print(reasoning)
    print("-" * 90)

CLUE: Fortune tellers scathing about Oscar
ANSWER: TAROT

1. similarity = 0.9062
TART (scathing) about O[scar] (phonetic alphabet)
------------------------------------------------------------------------------------------
2. similarity = 0.9033
*(GIVE GEAR) (*out)
------------------------------------------------------------------------------------------
3. similarity = 0.8340
BAKER (cook) taking E (drug)
------------------------------------------------------------------------------------------


## 14. Try your own clue, answer and reasoning

Edit these three variables and rerun the cell.

In [17]:
my_clue = "Oscar managed to acquire good instrument"
my_answer = "ORGAN"
my_reasoning = "Example reasoning goes here"

score = reasoning_similarity(
    my_clue,
    my_answer,
    my_reasoning
)

print("Similarity:", score)

Similarity: 0.8828125


## What this notebook does — and does not do

### It does:
- use the original pretrained DeBERTa
- show tokenization
- expose hidden states
- create simple pooled embeddings
- compare text representations with cosine similarity

### It does not:
- train DeBERTa
- fine-tune any weights
- learn what a correct cryptic-crossword reasoning is
- produce a calibrated correctness score

If you later want a true **reasoning scorer**, that requires some additional training signal
or another pretrained model that was already trained specifically for verification/ranking.

# 15. Large-scale evaluation on many examples

This section evaluates the pretrained DeBERTa model on many validation examples,
still with **no fine-tuning**.

For each example we compare:

- similarity between `CLUE + ANSWER` and the correct reasoning
- similarity between the same `CLUE + ANSWER` and a wrong reasoning

The main metric is **pairwise accuracy**:

\[
P(S_{correct} > S_{wrong})
\]

A random baseline is approximately 50%.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib

# Batch jobs have no display, so render figures to files.
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG_DIR = os.environ.get("FIG_DIR", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

EVAL_SEED = 42
N_EVAL = min(1000, len(dataset["validation"]))
BATCH_SIZE = 32

print("Validation-set size:", len(dataset["validation"]))
print("Examples used for evaluation:", N_EVAL)
print("Figures will be written to:", os.path.abspath(FIG_DIR))

## 15.1 Build the evaluation set

In [19]:
rng = random.Random(EVAL_SEED)

all_indices = list(range(len(dataset["validation"])))
rng.shuffle(all_indices)
eval_indices = all_indices[:N_EVAL]

rows = []

for idx in eval_indices:
    ex = dataset["validation"][idx]

    rows.append({
        "dataset_index": idx,
        "clue": clean_clue(ex["clue"]),
        "answer": str(ex["answer"]),
        "reasoning": str(ex["wordplay"]),
    })

eval_df = pd.DataFrame(rows)

print(eval_df.head())
print("\nNumber of evaluation examples:", len(eval_df))

   dataset_index                                      clue    answer  \
0            106         Large amount of money in mailboat    PACKET   
1            239  Sample duck taken from kitchen appliance    TASTER   
2             37     Condemn old banger to bottom of river  EXECRATE   
3             89                         Bob gets the bird      DUCK   
4             32       Murdoch maybe used to control pupil      IRIS   

                                           reasoning  
0                                  Double Definition  
1  T(o)ASTER (kitchen appliance, O (duck) taken f...  
2        CRATE (old banger) to bottom of EXE (river)  
3                                  Double Definition  
4                                  Double Definition  

Number of evaluation examples: 300


## 15.2 Create wrong reasonings

In [20]:
correct_reasonings = eval_df["reasoning"].tolist()

# Shift by one position so every example receives another example's reasoning.
wrong_reasonings = correct_reasonings[1:] + correct_reasonings[:1]

eval_df["wrong_reasoning"] = wrong_reasonings

print("Example pair:")
print("CLUE:", eval_df.iloc[0]["clue"])
print("ANSWER:", eval_df.iloc[0]["answer"])
print("\nCORRECT:")
print(eval_df.iloc[0]["reasoning"])
print("\nWRONG:")
print(eval_df.iloc[0]["wrong_reasoning"])

Example pair:
CLUE: Large amount of money in mailboat
ANSWER: PACKET

CORRECT:
Double Definition

WRONG:
T(o)ASTER (kitchen appliance, O (duck) taken from)


## 15.3 Batched embedding function

In [21]:
def get_embeddings_batch(texts, batch_size=32, max_length=256):
    embeddings = []

    model.eval()

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        hidden = outputs.last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1)

        summed = (hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)

        batch_embeddings = summed / counts
        batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)

        embeddings.append(batch_embeddings.cpu())

    return torch.cat(embeddings, dim=0)

## 15.4 Run DeBERTa on all evaluation texts

In [22]:
contexts = [
    f"CLUE: {row.clue}\nANSWER: {row.answer}"
    for row in eval_df.itertuples()
]

context_embeddings = get_embeddings_batch(
    contexts,
    batch_size=BATCH_SIZE,
)

correct_embeddings = get_embeddings_batch(
    eval_df["reasoning"].tolist(),
    batch_size=BATCH_SIZE,
)

wrong_embeddings = get_embeddings_batch(
    eval_df["wrong_reasoning"].tolist(),
    batch_size=BATCH_SIZE,
)

correct_scores = (context_embeddings * correct_embeddings).sum(dim=1).numpy()
wrong_scores = (context_embeddings * wrong_embeddings).sum(dim=1).numpy()

eval_df["correct_score"] = correct_scores
eval_df["wrong_score"] = wrong_scores
eval_df["margin"] = eval_df["correct_score"] - eval_df["wrong_score"]

print("Finished scoring", len(eval_df), "examples.")

Finished scoring 300 examples.


# 15.5 FINAL RESULTS

In [23]:
pairwise_accuracy = (
    eval_df["correct_score"] > eval_df["wrong_score"]
).mean()

ties = (
    eval_df["correct_score"] == eval_df["wrong_score"]
).mean()

mean_correct = eval_df["correct_score"].mean()
mean_wrong = eval_df["wrong_score"].mean()
mean_margin = eval_df["margin"].mean()
median_margin = eval_df["margin"].median()

print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)
print(f"Number of examples:       {len(eval_df)}")
print(f"Pairwise accuracy:        {pairwise_accuracy:.2%}")
print(f"Random baseline:          50.00%")
print(f"Ties:                     {ties:.2%}")
print()
print(f"Mean correct similarity:  {mean_correct:.4f}")
print(f"Mean wrong similarity:    {mean_wrong:.4f}")
print(f"Mean margin:              {mean_margin:.4f}")
print(f"Median margin:            {median_margin:.4f}")
print("=" * 70)

FINAL RESULTS
Number of examples:       300
Pairwise accuracy:        56.33%
Random baseline:          50.00%
Ties:                     2.33%

Mean correct similarity:  0.8701
Mean wrong similarity:    0.8623
Mean margin:              0.0075
Median margin:            0.0100


## 15.6 95% bootstrap confidence interval

In [24]:
rng_np = np.random.default_rng(EVAL_SEED)

wins = (
    eval_df["correct_score"].to_numpy()
    >
    eval_df["wrong_score"].to_numpy()
).astype(float)

bootstrap_acc = []

for _ in range(2000):
    sample = rng_np.choice(wins, size=len(wins), replace=True)
    bootstrap_acc.append(sample.mean())

ci_low, ci_high = np.percentile(
    bootstrap_acc,
    [2.5, 97.5]
)

print(f"Pairwise accuracy: {pairwise_accuracy:.2%}")
print(f"95% bootstrap CI:  [{ci_low:.2%}, {ci_high:.2%}]")

Pairwise accuracy: 56.33%
95% bootstrap CI:  [50.99%, 61.67%]


## 15.7 Score distributions

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(eval_df["correct_score"], bins=30, alpha=0.6, label="correct")
plt.hist(eval_df["wrong_score"], bins=30, alpha=0.6, label="wrong")
plt.xlabel("Cosine similarity")
plt.ylabel("Number of examples")
plt.title("Correct vs wrong reasoning similarity")
plt.legend()
plt.savefig(f"{FIG_DIR}/score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 15.8 Margin distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(eval_df["margin"], bins=30)
plt.axvline(0, linestyle="--")
plt.xlabel("Correct score - wrong score")
plt.ylabel("Number of examples")
plt.title("Pairwise margin distribution")
plt.savefig(f"{FIG_DIR}/margin_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 15.9 Best and worst examples

In [27]:
print("TOP 5 SUCCESSES")
print("=" * 100)

for _, row in eval_df.sort_values("margin", ascending=False).head(5).iterrows():
    print("\nCLUE:", row["clue"])
    print("ANSWER:", row["answer"])
    print(f"Correct score: {row['correct_score']:.4f}")
    print("Correct reasoning:", row["reasoning"])
    print(f"Wrong score:   {row['wrong_score']:.4f}")
    print("Wrong reasoning:", row["wrong_reasoning"])
    print(f"Margin: {row['margin']:.4f}")
    print("-" * 100)

print("\n\nTOP 5 FAILURES")
print("=" * 100)

for _, row in eval_df.sort_values("margin", ascending=True).head(5).iterrows():
    print("\nCLUE:", row["clue"])
    print("ANSWER:", row["answer"])
    print(f"Correct score: {row['correct_score']:.4f}")
    print("Correct reasoning:", row["reasoning"])
    print(f"Wrong score:   {row['wrong_score']:.4f}")
    print("Wrong reasoning:", row["wrong_reasoning"])
    print(f"Margin: {row['margin']:.4f}")
    print("-" * 100)

TOP 5 SUCCESSES

CLUE: Tragically misreads guns
ANSWER: SIDE ARMS
Correct score: 0.8774
Correct reasoning: (MISREADS)* (*tragically)
Wrong score:   0.5928
Wrong reasoning: S[w]E[e]T [j]U[m]P (oddly)
Margin: 0.2847
----------------------------------------------------------------------------------------------------

CLUE: Condemn old banger to bottom of river
ANSWER: EXECRATE
Correct score: 0.9229
Correct reasoning: CRATE (old banger) to bottom of EXE (river)
Wrong score:   0.6899
Wrong reasoning: Double Definition
Margin: 0.2329
----------------------------------------------------------------------------------------------------

CLUE: Mature player originally employed by French FA
ANSWER: RIPEN
Correct score: 0.9062
Correct reasoning: P[layer] (originally) emplyed by RIEN (French FA)
Wrong score:   0.6919
Wrong reasoning: G[l]E[a]N [p]I[e]C[e] (regularly)
Margin: 0.2144
----------------------------------------------------------------------------------------------------

CLUE: Tend to be

## How to interpret the final result

The key number is **Pairwise accuracy**.

- ~50%: the similarity heuristic is behaving about like random ranking.
- clearly above 50%: pretrained DeBERTa representations contain some useful signal.
- below 50%: this particular heuristic is not useful for the task.

Remember: no fine-tuning is performed here.
This tests only the pretrained model's representations plus mean pooling and cosine similarity.

## 16. Compare `CLUE` with `ANSWER + REASONING`

In [28]:
def clue_answer_reasoning_similarity(clue, answer, reasoning):
    clue_text = clean_clue(clue)
    answer_reasoning_text = f"ANSWER: {answer}\nREASONING: {reasoning}"

    return cosine_similarity(
        clue_text,
        answer_reasoning_text
    )

clue_embeddings = get_embeddings_batch(
    eval_df["clue"].tolist(),
    batch_size=BATCH_SIZE,
)

correct_answer_reasoning_texts = [
    f"ANSWER: {row.answer}\nREASONING: {row.reasoning}"
    for row in eval_df.itertuples()
]
correct_ar_embeddings = get_embeddings_batch(
    correct_answer_reasoning_texts,
    batch_size=BATCH_SIZE,
)

wrong_answer_reasoning_texts = [
    f"ANSWER: {row.answer}\nREASONING: {row.wrong_reasoning}"
    for row in eval_df.itertuples()
]
wrong_ar_embeddings = get_embeddings_batch(
    wrong_answer_reasoning_texts,
    batch_size=BATCH_SIZE,
)

correct_ar_scores = (clue_embeddings * correct_ar_embeddings).sum(dim=1).numpy()
wrong_ar_scores = (clue_embeddings * wrong_ar_embeddings).sum(dim=1).numpy()

eval_df["clue_ar_correct_score"] = correct_ar_scores
eval_df["clue_ar_wrong_score"] = wrong_ar_scores
eval_df["clue_ar_margin"] = eval_df["clue_ar_correct_score"] - eval_df["clue_ar_wrong_score"]

print("Finished scoring CLUE vs ANSWER + REASONING for", len(eval_df), "examples.")

Finished scoring CLUE vs ANSWER + REASONING for 300 examples.


## 16.1 FINAL RESULTS: `CLUE` vs `ANSWER + REASONING`

In [29]:
pairwise_accuracy_clue_ar = (
    eval_df["clue_ar_correct_score"] > eval_df["clue_ar_wrong_score"]
).mean()

ties_clue_ar = (
    eval_df["clue_ar_correct_score"] == eval_df["clue_ar_wrong_score"]
).mean()

mean_correct_clue_ar = eval_df["clue_ar_correct_score"].mean()
mean_wrong_clue_ar = eval_df["clue_ar_wrong_score"].mean()
mean_margin_clue_ar = eval_df["clue_ar_margin"].mean()
median_margin_clue_ar = eval_df["clue_ar_margin"].median()

print("=" * 70)
print("FINAL RESULTS: CLUE vs ANSWER + REASONING")
print("=" * 70)
print(f"Number of examples:       {len(eval_df)}")
print(f"Pairwise accuracy:        {pairwise_accuracy_clue_ar:.2%}")
print(f"Random baseline:          50.00%")
print(f"Ties:                     {ties_clue_ar:.2%}")
print()
print(f"Mean correct similarity:  {mean_correct_clue_ar:.4f}")
print(f"Mean wrong similarity:    {mean_wrong_clue_ar:.4f}")
print(f"Mean margin:              {mean_margin_clue_ar:.4f}")
print(f"Median margin:            {median_margin_clue_ar:.4f}")
print("=" * 70)

FINAL RESULTS: CLUE vs ANSWER + REASONING
Number of examples:       300
Pairwise accuracy:        54.33%
Random baseline:          50.00%
Ties:                     1.33%

Mean correct similarity:  0.8457
Mean wrong similarity:    0.8428
Mean margin:              0.0031
Median margin:            0.0039
